# FraudShield Colab Training Notebook

This notebook uses **Unsloth + TRL** to fine-tune a small instruction model to imitate strong investigation trajectories in FraudShield.

It is designed for **reliable Colab execution** first: install dependencies, build a training set from real environment rollouts, fine-tune a LoRA policy, evaluate heuristic vs trained policy, and save the expected training artifacts.


In [ ]:
%pip uninstall -y unsloth unsloth_zoo trl transformers tokenizers
%pip install -q openenv-core datasets peft accelerate sentencepiece
%pip install -q "transformers==4.51.3" "trl==0.19.1"
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

%cd /content
!rm -rf Fraudshield
!git clone https://github.com/DevikaJ2005/Fraudshield.git
%cd /content/Fraudshield
!ls
%pip install -q -e .


In [ ]:
import os
from getpass import getpass

from huggingface_hub import login

token = getpass('Enter your HF token (optional but recommended): ')
if token.strip():
    os.environ['HF_TOKEN'] = token.strip()
    login(token=token.strip())
    print('HF login completed.')
else:
    print('Skipping HF login for now.')


In [ ]:
import torch

print('cuda available:', torch.cuda.is_available())
print('device count:', torch.cuda.device_count())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('GPU not available. In Colab, set Runtime > Change runtime type > GPU, then restart.')


In [ ]:
import json
import os
import subprocess
from datetime import datetime

from datasets import Dataset

from fraudshield_env import FraudShieldEnvironment
from llm_agent import SnapshotCalibratedFraudDetectionAgent

env = FraudShieldEnvironment(data_path='data', seed=42)
assert env.load_data(), 'FraudShield snapshot failed to load.'
print('FraudShield loaded:', env.data_loader.get_bundle_summary())

def serialize_observation(observation):
    return json.dumps(
        {
            'case_id': observation.case_id,
            'task_name': observation.task_name.value,
            'current_screen': observation.current_screen.value,
            'visible_panels': observation.visible_panels,
            'revealed_evidence': observation.revealed_evidence,
            'linked_case_ids': observation.linked_case_ids,
            'remaining_steps': observation.remaining_steps,
            'remaining_sla': observation.remaining_sla,
            'note_required': observation.note_required,
            'allowed_actions': [action.value for action in observation.allowed_actions],
            'case_summary': observation.case_summary.model_dump(mode='json'),
            'app_context': observation.app_context,
        },
        ensure_ascii=True,
        indent=2,
    )

def prompt_from_observation(observation):
    return (
        'You are a fraud analyst working in a simulated investigation workflow.\n'
        'Choose the next best action based only on the visible observation.\n'
        'Respond with JSON only using keys action_type, investigation_target, decision, confidence, reasoning.\n'
        'Use action_type investigate or decide.\n\n'
        'Observation:\n'
        f"{serialize_observation(observation)}\n"
    )

def action_to_target_json(action):
    payload = {
        'action_type': 'investigate',
        'investigation_target': None,
        'decision': None,
        'confidence': 0.5,
        'reasoning': action.reasoning or '',
    }
    if action.action_type.value == 'fetch_customer_profile':
        payload['investigation_target'] = 'customer_profile'
    elif action.action_type.value == 'fetch_merchant_profile':
        payload['investigation_target'] = 'merchant_profile'
    elif action.action_type.value == 'fetch_network_graph':
        payload['investigation_target'] = 'network_graph'
    elif action.action_type.value == 'check_policy':
        payload['investigation_target'] = 'policy_review'
    elif action.action_type.value == 'review_transaction':
        payload['investigation_target'] = 'payment_trace'
    elif action.action_type.value == 'add_case_note':
        payload['investigation_target'] = 'trust_notes'
        payload['reasoning'] = action.note_text or payload['reasoning']
    elif action.action_type.value == 'resolve_case':
        payload['action_type'] = 'decide'
        payload['investigation_target'] = None
        if action.resolution.value in {'approve', 'request_docs'}:
            payload['decision'] = 'legitimate'
            payload['confidence'] = 0.8 if action.resolution.value == 'approve' else 0.6
        else:
            payload['decision'] = 'fraud'
            payload['confidence'] = 0.9 if action.resolution.value in {'block', 'escalate'} else 0.6
    return json.dumps(payload, ensure_ascii=True)

def build_training_dataset(per_task_episodes=18):
    agent = SnapshotCalibratedFraudDetectionAgent()
    records = []
    for task_name in ('easy', 'medium', 'hard'):
        for episode_idx in range(per_task_episodes):
            rollout_env = FraudShieldEnvironment(data_path='data', seed=42 + episode_idx)
            rollout_env.load_data()
            observation = rollout_env.reset(task_name).observation
            while not rollout_env.is_done:
                action = agent.decide(observation)
                records.append({
                    'task_name': task_name,
                    'prompt': prompt_from_observation(observation),
                    'target': action_to_target_json(action),
                    'text': prompt_from_observation(observation) + action_to_target_json(action),
                })
                observation = rollout_env.step(action).observation
    return Dataset.from_list(records)

train_dataset = build_training_dataset(per_task_episodes=18)
print(train_dataset)
print(train_dataset[0]['text'][:1200])


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
)
print('Loaded base model and LoRA adapters.')


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='fraudshield-sft-run',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='epoch',
    report_to='none',
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_steps=-1,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

trainer.train()

OUTPUT_DIR = 'trained_policy'
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('Saved trained policy to', OUTPUT_DIR)


In [ ]:
def run_inference(extra_env=None):
    env_vars = os.environ.copy()
    if extra_env:
        env_vars.update(extra_env)
    completed = subprocess.run(
        ['python', 'inference.py'],
        capture_output=True,
        text=True,
        env=env_vars,
        check=True,
    )
    with open('fraudshield_baseline_results.json', 'r', encoding='utf-8') as handle:
        return json.load(handle), completed.stdout

baseline_results, baseline_stdout = run_inference()
trained_results, trained_stdout = run_inference({'LOCAL_MODEL_PATH': OUTPUT_DIR})

comparison_rows = []
for task_name in ('easy', 'medium', 'hard'):
    comparison_rows.append({
        'task': task_name,
        'heuristic_score': baseline_results[task_name]['score'],
        'trained_score': trained_results[task_name]['score'],
        'delta': trained_results[task_name]['score'] - baseline_results[task_name]['score'],
    })

print('Heuristic baseline stdout:\n', baseline_stdout)
print('Trained model stdout:\n', trained_stdout)
print(json.dumps(comparison_rows, indent=2))


In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
loss_points = [(entry['step'], entry['loss']) for entry in log_history if 'step' in entry and 'loss' in entry]

if loss_points:
    xs, ys = zip(*loss_points)
    plt.figure(figsize=(8, 4))
    plt.plot(xs, ys)
    plt.xlabel('training step')
    plt.ylabel('loss')
    plt.title('FraudShield loss curve')
    plt.tight_layout()
    plt.savefig('loss_curve.png')
    plt.close()

reward_like_points = [(idx + 1, row['delta']) for idx, row in enumerate(comparison_rows)]
plt.figure(figsize=(8, 4))
plt.plot([x for x, _ in reward_like_points], [y for _, y in reward_like_points], marker='o')
plt.xticks([1, 2, 3], ['easy', 'medium', 'hard'])
plt.xlabel('task')
plt.ylabel('score delta vs heuristic')
plt.title('FraudShield score improvement by task')
plt.tight_layout()
plt.savefig('reward_curve.png')
plt.close()

summary = {
    'status': 'completed',
    'updated_at': datetime.utcnow().isoformat() + 'Z',
    'trainer': 'TRL SFTTrainer with Unsloth LoRA',
    'base_model': MODEL_NAME,
    'local_model_path': OUTPUT_DIR,
    'baseline': {
        'easy': baseline_results['easy']['score'],
        'medium': baseline_results['medium']['score'],
        'hard': baseline_results['hard']['score'],
        'final_score': baseline_results['final_score'],
    },
    'trained': {
        'easy': trained_results['easy']['score'],
        'medium': trained_results['medium']['score'],
        'hard': trained_results['hard']['score'],
        'final_score': trained_results['final_score'],
    },
    'score_delta': trained_results['final_score'] - baseline_results['final_score'],
    'task_comparison': comparison_rows,
    'artifact_urls': {
        'reward_plot': 'reward_curve.png',
        'loss_plot': 'loss_curve.png',
        'comparison_table': 'training_summary.json',
    },
}

with open('training_summary.json', 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)

print(json.dumps(summary, indent=2))
